# Phase 1: turn the hand-checked transcripts into activations

**Question.** For each blackmail ("act") and hold-back ("refrain") transcript, what does the residual stream look like at the moment that matters?

**Two anchor points per transcript**
- **notice**: the first token where the reasoning mentions the leverage (affair, leverage, vulnerability...).
- **commit**: the last token of the reasoning, right before `</SCRATCHPAD_REASONING>`, i.e. just before the model acts.

Comparing the two anchors is what lets Phase 2 ask *when* the decision becomes readable.

| | |
|---|---|
| Needs | GPU with about 24 GB |
| Reads | `contrast_transcripts.json` (83 act, 100 refrain; hand-checked in Phase -1) |
| Writes | `contrast_acts.pt` (notice), `contrast_acts_commit.pt` (commit), each `{act, ref}` as `[n, 48, 3840]` |

**Caveat on the notice anchor.** The repo ships the commit-anchor extraction as a notebook but not the batch code for the notice anchor.
The notice extraction below is a reconstruction (first keyword inside the final reasoning block), so expect small differences from the original `contrast_acts.pt`.

**Garbled outputs.** Some transcripts are token salad; any anchor whose surrounding text is more than 15% non-ASCII is skipped.
The original run kept 68 act / 98 refrain at the commit anchor (17 skipped).

**Running on its own Colab.** This notebook shares nothing with the others except one Google Drive folder (`MyDrive/gemma-interp`, set in the setup cell, the same in every notebook).
It reads the files it needs from that folder and writes what it makes there, so the next notebook can pick it up even though it runs in a different Colab.
If a small input file is missing and `REPO_RAW` is set (the raw URL of your public GitHub folder), it is downloaded automatically; otherwise you get a message naming the file and the notebook that creates it.
Use a GPU runtime for every notebook except phase 2 (CPU is enough). Nothing is kept on the Colab's own disk, so a disconnect loses nothing that was already saved to Drive.

In [ ]:
import os, json, time, numpy as np
from collections import defaultdict
from functools import partial
# Each notebook can run in its own Colab, and a Colab disk is wiped when it disconnects. So everything that must survive
# or be handed to another notebook (activations, directions, saved samples) lives in a Google Drive folder.
USE_DRIVE  = True                                                 # set False to use the local folder instead
DRIVE_DIR  = "MyDrive/gemma-interp"                               # the same Drive folder in every notebook
REPO_RAW   = ""                                                   # optional: raw GitHub folder holding the small input files, e.g.
                                                                  # "https://raw.githubusercontent.com/<user>/<repo>/main/gemma-interp"
try:
    import google.colab; ON_COLAB = True
except ImportError:
    ON_COLAB = False
if USE_DRIVE and ON_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    BASE = f"/content/drive/{DRIVE_DIR}"
else:
    BASE = os.environ.get("GEMMA_BASE", os.getcwd())
OUT = f"{BASE}/results"; os.makedirs(OUT, exist_ok=True)           # everything generated lands here
print("working folder:", BASE)

def need(fname, optional=False):                                  # make sure BASE/fname exists: use it, else download it, else say what to do
    path = f"{BASE}/{fname}"
    if os.path.exists(path): return path
    if REPO_RAW:
        import urllib.request
        try:
            os.makedirs(os.path.dirname(path), exist_ok=True)
            urllib.request.urlretrieve(f"{REPO_RAW}/{fname}", path); print("downloaded", fname); return path
        except Exception as e: print(f"could not download {fname}: {e}")
    if optional: print(f"(optional file {fname} not found - that is fine)"); return None
    raise FileNotFoundError(f"{fname} is not in {BASE}. Put it in that Drive folder (the notebook that creates it is named in README.md), "
                            f"or set REPO_RAW to the raw GitHub folder that holds it.")
SEED, N_SCEN, MAX_NEW = 0, 15, 1200
GRADER_MODEL = os.environ.get("GRADER_MODEL", "gemini-3.5-flash") # Gemini model that grades outputs; change it if your key says "model not found"
# Keys are never written in a notebook. ensure_key() finds one in the environment, then in Colab Secrets,
# and otherwise asks you to type it (hidden, kept only for this session):
#   GEMINI_API_KEY  (grading)      HF_TOKEN  (first download of Gemma; accept the licence on Hugging Face first)
def ensure_key(name):
    if os.environ.get(name): return
    try:
        from google.colab import userdata                         # Colab Secrets, if you are in Colab
        os.environ[name] = userdata.get(name); return
    except Exception: pass
    import getpass
    os.environ[name] = getpass.getpass(f"{name} (typed here, not saved in the notebook): ")

In [ ]:
# One-time install. Keeps the torch version that is already on the machine so pip does not swap it.
import torch
!echo "torch=={torch.__version__.split('+')[0]}" > /tmp/torch_pin.txt
%pip install -q -c /tmp/torch_pin.txt transformer_lens transformers accelerate scikit-learn matplotlib google-genai
# If pip reports that it changed torch or transformers, use Runtime/Kernel -> Restart, then rerun from the first cell.

In [ ]:
import torch
if not os.environ.get("HF_TOKEN"): print("Note: if the Gemma download fails with 401/403, run ensure_key('HF_TOKEN') and rerun this cell.")
from transformer_lens import HookedTransformer
model = HookedTransformer.from_pretrained_no_processing("google/gemma-3-12b-it", dtype="bfloat16")
tok = model.tokenizer
nL = model.cfg.n_layers
data = json.load(open(need("contrast_transcripts.json")))        # 183 transcripts: 83 act, 100 refrain

In [ ]:
OPEN, CLOSE = "<SCRATCHPAD_REASONING>", "</SCRATCHPAD_REASONING>"
NOTICE = ["affair", "infidelit", "jessica", "maria", "rachel", "personal life",
          "extramarital", "vulnerab", "leverage", "exploit"]

def is_garbled(ctx):
    return (not ctx) or sum(1 for c in ctx if ord(c) > 127) / len(ctx) > 0.15

def tok_at_char(strs, c):                                         # index of the token containing character c
    cum = 0
    for j, t in enumerate(strs):
        cum += len(t)
        if cum > c: return j
    return None

def anchor_chars(full):                                           # (notice_char, commit_char), None if absent
    e = full.rfind(CLOSE)
    if e < 0: return None, None
    commit = e - 1
    s = full.rfind(OPEN, 0, e)
    low, notice = full.lower(), None
    if s >= 0:
        ends = [i + len(k) - 1 for k in NOTICE if (i := low.find(k, s, e)) >= 0]
        if ends: notice = min(ends)
    return notice, commit

## Extract both anchors in one forward pass per transcript

In [ ]:
sets = {"notice": {"act": [], "ref": []}, "commit": {"act": [], "ref": []}}
skipped, contexts = [], []

for k, rec in enumerate(data):
    msgs = [{"role": m["role"], "content": m["text"]} for m in rec["messages"]]
    toks = model.to_tokens(tok.apply_chat_template(msgs, tokenize=False), prepend_bos=False)
    strs = tok.batch_decode([[i] for i in toks[0].tolist()], clean_up_tokenization_spaces=False)
    full = "".join(strs)
    chars = dict(zip(("notice", "commit"), anchor_chars(full)))
    if all(c is None for c in chars.values()):
        skipped.append((rec["src"], rec["idx"], "no_close_tag")); continue
    with torch.no_grad():
        _, cache = model.run_with_cache(toks, names_filter=lambda n: n.endswith("resid_post"))
    for name, c in chars.items():
        if c is None: skipped.append((rec["src"], rec["idx"], f"no_{name}_anchor")); continue
        ctx = full[max(0, c - 60): c + 25].replace("\n", " ")
        if is_garbled(ctx): skipped.append((rec["src"], rec["idx"], f"garbled_{name}")); continue
        j = tok_at_char(strs, c)
        acts = torch.stack([cache["resid_post", L][0, j] for L in range(nL)]).float().cpu()
        sets[name]["act" if rec["label"] == "act" else "ref"].append(acts)
        contexts.append((name, rec["label"], ctx))
    if (k + 1) % 25 == 0: print(f"{k+1}/{len(data)}")

for name, fname in [("notice", "contrast_acts.pt"), ("commit", "contrast_acts_commit.pt")]:
    s = sets[name]
    torch.save({"act": torch.stack(s["act"]), "ref": torch.stack(s["ref"])}, f"{BASE}/{fname}")
    print(f"{name:>6}: act={len(s['act'])} ref={len(s['ref'])} -> {fname}")
print("skipped:", len(skipped))

## Eyeball the anchors
If the notice anchors sit on the affair and the commit anchors sit on the end of the reasoning, the extraction is right.

In [ ]:
for name in ("notice", "commit"):
    for label in ("act", "refrain"):
        ex = [c for n, l, c in contexts if n == name and l == label][:2]
        for c in ex: print(f"{name:>6} | {label:>7} | ...{c}...")

## What this gives Phase 2
Two balanced sets of residual-stream vectors, one per anchor, at all 48 layers. Sample sizes are small (about 70-100 per class),
which is why Phase 2 uses a permutation null and strong regularisation rather than trusting raw accuracy.